
# Лабораторна робота №2  
1. **Завдання 1** — аналіз датасету **Global YouTube Statistics 2023**
2. **Завдання 2** — аналіз датасету **Amazon Top 50 Bestselling Books 2009–2019**

In [1]:

!pip -q install kagglehub

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'matplotlib'

## Завдання 1. 



In [ ]:

# Завантаження датасету
path1 = kagglehub.dataset_download("nelgiriyewithana/global-youtube-statistics-2023")
print("Path to dataset files:", path1)
print("Files in dataset folder:", os.listdir(path1))

csv_files1 = [f for f in os.listdir(path1) if f.endswith(".csv")]
print("CSV files:", csv_files1)

csv_file1 = os.path.join(path1, csv_files1[0])

youtube_df = pd.read_csv(csv_file1)

print("Перші 5 рядків датасету:")
display(youtube_df.head())

print("Розміри датасету:", youtube_df.shape)

print("Назви стовпців:")
print(list(youtube_df.columns))


In [ ]:


print("Кількість пропусків до обробки:")
print(youtube_df.isna().sum())


In [ ]:

youtube_clean = youtube_df.copy()

for col in youtube_clean.columns:
    if youtube_clean[col].dtype == 'object':
        youtube_clean[col] = youtube_clean[col].astype(str).str.replace(',', '', regex=False)
        youtube_clean[col] = youtube_clean[col].replace(['nan', 'None', '', ' '], np.nan)

# Спробуємо перетворити ймовірно числові стовпці
for col in youtube_clean.columns:
    sample = youtube_clean[col].dropna().astype(str).head(20)
    numeric_like = sample.str.fullmatch(r'-?\d+(\.\d+)?').mean() if len(sample) > 0 else 0
    if numeric_like >= 0.6:
        youtube_clean[col] = pd.to_numeric(youtube_clean[col], errors='coerce')

print("Кількість пропусків після приведення до NaN та числових типів:")
print(youtube_clean.isna().sum())


In [ ]:


numeric_cols = youtube_clean.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    youtube_clean[col] = youtube_clean[col].fillna(youtube_clean[col].mean()).astype(float)

print("Числові стовпці після заповнення середнім:")
print(list(numeric_cols))


In [ ]:


print("Кількість пропусків після заповнення:")
print(youtube_clean.isna().sum())


In [ ]:


if 'Country' in youtube_clean.columns:
    unique_countries = youtube_clean['Country'].nunique(dropna=True)
    print("Кількість унікальних країн:", unique_countries)
else:
    print("Стовпець 'Country' не знайдено.")


In [ ]:


views_col = None
for c in youtube_clean.columns:
    if str(c).strip().lower() in ['video views', 'video_views', 'videoviews', 'views']:
        views_col = c
        break

print("Стовпець з переглядами:", views_col)


In [ ]:


if views_col:
    youtube_clean[views_col].plot(kind='hist', bins=30, figsize=(10, 5))
    plt.title('Розподіл переглядів')
    plt.xlabel('Кількість переглядів')
    plt.ylabel('Частота')
    plt.show()
else:
    print("Стовпець із переглядами не знайдено.")


In [ ]:


if views_col:
    print("Максимальна кількість переглядів:", youtube_clean[views_col].max())
    print("Мінімальна кількість переглядів:", youtube_clean[views_col].min())
    print("Середня кількість переглядів:", youtube_clean[views_col].mean())
else:
    print("Стовпець із переглядами не знайдено.")


In [ ]:


country_col = 'Country' if 'Country' in youtube_clean.columns else None

uploads_col = None
for c in youtube_clean.columns:
    if str(c).strip().lower() == 'uploads':
        uploads_col = c
        break

if country_col and uploads_col:
    country_uploads = youtube_clean.groupby(country_col)[uploads_col].sum().sort_values(ascending=False)
    print("Країна, де найбільше відео було завантажено на YouTube:")
    print(country_uploads.idxmax(), '-', country_uploads.max())
else:
    print("Не знайдено потрібні стовпці 'Country' та/або 'uploads'.")


In [ ]:


name_col = None
for c in youtube_clean.columns:
    if str(c).strip().lower() in ['youtuber', 'channel name', 'channel_name']:
        name_col = c
        break

if uploads_col and name_col:
    max_row = youtube_clean.loc[youtube_clean[uploads_col].idxmax()]
    min_row = youtube_clean.loc[youtube_clean[uploads_col].idxmin()]

    print("Канал з найбільшою кількістю uploads:")
    print(max_row[name_col], '-', max_row[uploads_col])

    print("\nКанал з найменшою кількістю uploads:")
    print(min_row[name_col], '-', min_row[uploads_col])
else:
    print("Не знайдено потрібні стовпці для визначення назви каналу або uploads.")


## Завдання 2. Amazon Top 50 Bestselling Books 2009–2019

In [ ]:


path2 = kagglehub.dataset_download("sootersaalu/amazon-top-50-bestselling-books-2009-2019")
print("Path to dataset files:", path2)
print("Files in dataset folder:", os.listdir(path2))

csv_files2 = [f for f in os.listdir(path2) if f.endswith(".csv")]
print("CSV files:", csv_files2)

csv_file2 = os.path.join(path2, csv_files2[0])

books_df = pd.read_csv(csv_file2)

print("Перші 10 рядків:")
display(books_df.head(10))


print("Розміри датасету:", books_df.shape)
print("Датасет містить дані про таку кількість книг:", books_df.shape[0])


In [ ]:


books_df.columns = ['name', 'author', 'user_rating', 'reviews', 'price', 'year', 'genre']
print("Нові назви стовпців:")
print(books_df.columns.tolist())


In [ ]:

missing_books = books_df.isna().sum()
print("Кількість пропусків:")
print(missing_books)

if missing_books.sum() == 0:
    print("Чи є в якихось змінних пропуски? Ні")
else:
    print("Чи є в якихось змінних пропуски? Так")


In [ ]:

unique_genres = books_df['genre'].unique()
print("Унікальні жанри:")
print(unique_genres)


In [ ]:

print("Максимальна ціна:", books_df['price'].max())
print("Мінімальна ціна:", books_df['price'].min())
print("Середня ціна:", books_df['price'].mean())
print("Медіанна ціна:", books_df['price'].median())


In [ ]:

max_rating = books_df['user_rating'].max()
print("Найвищий рейтинг у датасеті:", max_rating)

count_max_rating = books_df[books_df['user_rating'] == max_rating].shape[0]
print("Скільки книг мають такий рейтинг:", count_max_rating)

book_max_reviews = books_df.loc[books_df['reviews'].idxmax(), 'name']
print("Книга, яка має найбільше відгуків:", book_max_reviews)

most_expensive_2010 = books_df[books_df['year'] == 2010].sort_values(by='price', ascending=False).head(1)
print("Найдорожча книга серед Топ-50 у 2010 році:")
display(most_expensive_2010[['name', 'price']])

fiction_2012_count = books_df[(books_df['genre'] == 'Fiction') & (books_df['year'] == 2012)].shape[0]
print("Кількість книг жанру Fiction у 2012 році:", fiction_2012_count)

rating_49_count = books_df[(books_df['user_rating'] == 4.9) & (books_df['year'].isin([2010, 2011]))].shape[0]
print("Кількість книг з рейтингом 4.9 у 2010 та 2011 роках:", rating_49_count)


In [ ]:

genre_price = books_df.groupby('genre')[['price']].agg(['min', 'max'])
print("Мінімальна і максимальна ціна для кожного жанру:")
display(genre_price)


In [ ]:

youtube_clean.to_csv('youtube_cleaned.csv', index=False)
books_df.to_csv('books_cleaned.csv', index=False)

print("Файли youtube_cleaned.csv та books_cleaned.csv збережено.")



## Висновок

У ході лабораторної роботи було виконано аналіз двох CSV-датасетів